In [ ]:
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt 

car = pd.read_csv('quikr_car.csv')


# AUTO‑GENERATE VALID YEAR RANGE FOR EACH CAR NAME
year_rules = {}

for name, group in car.groupby("name"):
    year_rules[name] = (int(group["year"].min()), int(group["year"].max()))

# Convert non-numeric values to NaN first
car['year'] = pd.to_numeric(car['year'], errors='coerce')

# Convert valid years to int
car['year'] = car['year'].astype('Int64')

# Remove "Ask For Price"
car['Price'] = car['Price'].str.replace('Ask For Price', '', regex=False)

# Replace empty-like values with NaN
car['Price'] = car['Price'].replace(['', ' ', '  ', '-', '--'], np.nan)

# Remove commas
car['Price'] = car['Price'].str.replace(',', '', regex=False)

# Convert to numeric (invalid → NaN)
car['Price'] = pd.to_numeric(car['Price'], errors='coerce')

# Fill NaN with mean price
car['Price'] = car['Price'].fillna(car['Price'].mean())

# Round prices
car['Price'] = car['Price'].astype(float).round(2)

# Cleaning Kms_driven
car['kms_driven'] = (
    car['kms_driven']
    .astype(str)
    .str.replace(',', '', regex=False)
    .str.replace('kms', '', regex=False)
    .str.replace('km', '', regex=False)
    .str.replace('Petrol', '', regex=False)
    .str.strip()
    .replace('', np.nan)
)

car['kms_driven'] = pd.to_numeric(car['kms_driven'], errors='coerce')
car['kms_driven'] = car['kms_driven'].fillna(car['kms_driven'].median())
car['kms_driven'] = car['kms_driven'].astype(int)
car['kms_driven']

# Fixing the null values 
car = car[~car['fuel_type'].isna()]

# taking the first 3 word of name
car['name'] = car['name'].str.split().str[:3].str.join(' ')

# Fixing the index 
car = car.reset_index(drop=True)

car = car[car['Price']<6e6].reset_index(drop=True)

# Model 
car.to_csv('clean_car.csv')
x = car.drop(columns=['Price'])  # features 
y = car['Price']             # target

from sklearn.model_selection import train_test_split
x_train, x_test, y_train, y_test = train_test_split(x,y,test_size=0.2)

from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import make_column_transformer
from sklearn.pipeline import make_pipeline

ohe = OneHotEncoder()
ohe.fit(x[['name','company','fuel_type']])

column_trans = make_column_transformer(
    (OneHotEncoder(handle_unknown='ignore'), ['name','company','fuel_type']),
    remainder='passthrough'
)


lr = LinearRegression()
pipe = make_pipeline(column_trans, lr)
pipe.fit(x_train,y_train)

y_pred = pipe.predict(x_test)
r2_score(y_test, y_pred)           #0.7235312384902404

# Trying to get maximum r2 score 
scores = []
for i in range(1000):
    x_train, x_test, y_train, y_test = train_test_split(x,y,test_size=0.2, random_state=i)
    lr = LinearRegression()
    pipe = make_pipeline(column_trans, lr)
    pipe.fit(x_train,y_train)
    y_pred = pipe.predict(x_test)
    scores.append(r2_score(y_test,y_pred))

scores[np.argmax(scores)]      # 0.8277663974479011

x_train, x_test, y_train, y_test = train_test_split(x,y,test_size=0.2, random_state=np.argmax(scores))
lr = LinearRegression()
pipe = make_pipeline(column_trans, lr)
pipe.fit(x_train,y_train)
y_pred = pipe.predict(x_test)
r2_score(y_test,y_pred)


0.8240483461382144

In [261]:
import pickle
pickle.dump(pipe, open('car_price_model.pkl','wb'))


In [ ]:
# Quality (info)
"""
year - object - blank values- mix data-special characters
price - object - commas -Outliers 
kms_driven - object mix (kms), commas, null, Outliers
fuel_type - null value 
company - numbers, scrapt words

"""
# stats (describe)
"""
price - outliers 

"""

'\nyear - object - blank values- mix data-special characters\nprice - object - commas -Outliers \nkms_driven - object mix (kms), commas, null, Outliers\nfuel_type - null value \ncompany - numbers, scrapt words\n\n'